### Taxonomic ID of DR sequence data

#### Kraken2

In [ ]:
#kraken2 already installed
/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/kraken2

#use DB downloaded by unity
/datasets/bio/kraken2/kraken2-db

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-kraken2-mcav-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/kraken2

SAMPLENAME="mcav"
DBNAME="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/ref_databases/standard"
READS="/scratch3/workspace/nikea_ulrich_uml_edu-data_download/DR_final_filtered/${SAMPLENAME}"
OUTDIR="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/kraken2"
mkdir -p $OUTDIR
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/ID_tables"
SAMPLELIST="DR_${SAMPLENAME}_sampleids.txt"

THREADS=20
KMER_LEN=35 #(default)
READ_LEN=150

# classify each set of paired end reads against refseq taxonomic database
while IFS= read -r SAMPLEID; do
kraken2 --db $DBNAME --threads $THREADS --report $OUTDIR/"${SAMPLEID}".kreport2 --report-zero-counts --paired $READS/"${SAMPLEID}"_host_removed_R1.tagged_filter.fastq.gz $READS/"${SAMPLEID}"_host_removed_R2.tagged_filter.fastq.gz > $OUTDIR/"${SAMPLEID}".kraken2
if [ $? -eq 0 ]; then
        echo "kraken2 completed successfully for sample: $SAMPLEID"
    else
        echo "kraken2 encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done < "$LISTPATH/${SAMPLELIST}"
echo "Kraken2: All samples processed successfully."

conda deactivate

# JOB-ID: 51628538
# bash script file name: /bash_scripts/DR_kraken2_mcav.sh

other job IDs: \
dcyl: 51628535 \
dlab: 51628536 \
ssid: 51628539, this failed (segmentation fault) memory was fine so idk \
pstr: 51628540, this failed (segmentation fault) memory was fine so idk \
mmea: 51628672

Didn't have the output folder being made so it wasn't running. Took me way to long to get that figured. Also ran out of memory from allocating 75G (was trying to get them queued faster but was canceled with OOM) so had to up memory and re-run

the unity kraken db (/datasets/bio/kraken2/kraken2-db) is not identifying like any sequences, so might need to stick with our /ref_databases/standard db

Even with the standard db it isn't identifying a lot, maybe re-pair the reads and see if that makes a difference? \
tried on mcav's to compare job ID: 51632944 \
This job failed like the others. It's a segmentation fault. It doesn't seem like it memory/RAM related because the node usage is below the 180 GB threshold. \
I'm going to assemble and then run them through kraken2 and see if that helps. 


Also having unity install new kraken2 databases: PrackenDB and PlusPF

#### trying this again on with new kraken DBs!

PrackenDB and Kraken2 PlusPF have been installed in /datasets/bio/kraken2/PrackenDB and /datasets/bio/kraken2/PlusPF respectively. 

You can use both databases with the following command: 
kraken2 classify --db  --threads $SLURM_CPUS_ON_NODE  

The PrackenDB is quite large (~500GB) so I would recommend to use the flag --memory-mapping to avoid loading the database into RAM.

In [ ]:
*not a working script**
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-kraken2_prackendb-mcav-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/kraken2

SAMPLENAME="mcav"
DBNAME="/datasets/bio/kraken2/PrackenDB"
READS="/scratch3/workspace/nikea_ulrich_uml_edu-data_download/DR_final_filtered/${SAMPLENAME}"
OUTDIR="/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/kraken2/${SAMPLENAME}"
mkdir -p $OUTDIR
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/ID_tables"
SAMPLELIST="DR_${SAMPLENAME}_sampleids.txt"


# classify each set of paired end reads against Pracken database, reads should come last (no other flags after --paired) 
while IFS= read -r SAMPLEID; do
kraken2 --db $DBNAME --threads $SLURM_CPUS_ON_NODE --report $OUTDIR/"${SAMPLEID}".kreport2 --report-zero-counts --memory-mapping --paired $READS/"${SAMPLEID}"_host_removed_R1.tagged_filter.fastq.gz $READS/"${SAMPLEID}"_host_removed_R2.tagged_filter.fastq.gz > $OUTDIR/"${SAMPLEID}".kraken2
if [ $? -eq 0 ]; then
        echo "kraken2 completed successfully for sample: $SAMPLEID"
    else
        echo "kraken2 encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done < "$LISTPATH/${SAMPLELIST}"
echo "Kraken2: All samples processed successfully."

conda deactivate

# JOB-ID: 52174555
# bash script file name: /bash_scripts/DR_kraken2_prackendb_mcav.sh

Job IDs: \
dcyl: 52185410 \

**Job ran for ~24hrs and then quite for mcav and dcyl when using PrackenDB. It didn't even get through one sample -- likely a memory issue even with the --memory-mapping flag. Proceeding with PlusPF database (see below) which worked. Still pretty low classification rate though. Will try running on fastqs from mapped bam files to see if that helps the classification! (see DR_Assembly.ipynb)**

**trying with the other db /datasets/bio/kraken2/PlusPF to compare, just running pstr (job ID: 52202302) and dcyl (job ID: 52202449)**

kept getting errors about: kraken2: --paired requires positive and even number filenames
kraken2 encountered an error for sample: 122021_Coralina_2021_Baya_027_PSTR4_S34

trying on repaired reads and removing the --log flag (this worked for pstr so proceeding with this approach) -got seq fault on dcyl samples after fulling running 4 samples, so will run on repaired reads and see if that helps (it does)

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-kraken2_pluspfdb-pstr-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/kraken2

SAMPLENAME="pstr"
DBNAME="/datasets/bio/kraken2/PlusPF"
READS="/scratch3/workspace/nikea_ulrich_uml_edu-data_download/DR_final_filtered/${SAMPLENAME}/repaired"
OUTDIR="/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/kraken2_pluspf/${SAMPLENAME}"
mkdir -p $OUTDIR
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/ID_tables"
SAMPLELIST="DR_${SAMPLENAME}_sampleids.txt"


# classify each set of paired end reads against Pracken database
while IFS= read -r SAMPLEID; do
kraken2 --db $DBNAME --threads $SLURM_CPUS_ON_NODE --report $OUTDIR/"${SAMPLEID}".kreport2 --report-zero-counts --memory-mapping --paired $READS/"${SAMPLEID}"_R1_ready.fastq.gz $READS/"${SAMPLEID}"_R2_ready.fastq.gz > $OUTDIR/"${SAMPLEID}".kraken2 
if [ $? -eq 0 ]; then
        echo "kraken2 completed successfully for sample: $SAMPLEID"
    else
        echo "kraken2 encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done < "$LISTPATH/${SAMPLELIST}"
echo "Kraken2: All samples processed successfully."

conda deactivate

# JOB-ID: 52235510
# bash script file name: /bash_scripts/DR_kraken2_pluspfdb_pstr.sh

other job ids: \
dcyl: 52235509 \
mcav: 52235512 \
ssid: 52237976 \
mmea: 52241488 \
dlab: 52237979

see DR_Assembly.ipynb for running kraken on reads from co-assemblies. \
Classification rate is pretty low so just trying running kraken on the RAW fastqs. Doing just for one MCAV sample for now to see what classification looks like. \
**it's higher, about double, but still only ~6%**

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-kraken2_pluspfdb-mcavraw-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/kraken2

#SAMPLENAME="pstr"
DBNAME="/datasets/bio/kraken2/PlusPF"
READS="/scratch3/workspace/nikea_ulrich_uml_edu-data_download/DR_Coralina_combined_raw"
OUTDIR="/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/kraken2_pluspf_mcav_raw"
mkdir -p $OUTDIR
#LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/ID_tables"
#SAMPLELIST="DR_${SAMPLENAME}_sampleids.txt"


# classify each set of paired end reads against Pracken database
#while IFS= read -r SAMPLEID; do
kraken2 --db $DBNAME --threads $SLURM_CPUS_ON_NODE --report $OUTDIR/072023_Carolina_2023_Baya_053_MCAV_S3.kreport2 --report-zero-counts --memory-mapping --paired $READS/072023_Carolina_2023_Baya_053_MCAV_S3_R1_001.fastq.gz $READS/072023_Carolina_2023_Baya_053_MCAV_S3_R2_001.fastq.gz > $OUTDIR/072023_Carolina_2023_Baya_053_MCAV_S3.kraken2 
#if [ $? -eq 0 ]; then
       # echo "kraken2 completed successfully for sample: $SAMPLEID"
   # else
     #   echo "kraken2 encountered an error for sample: $SAMPLEID"
     #   exit 1
   # fi
#done < "$LISTPATH/${SAMPLELIST}"
#echo "Kraken2: All samples processed successfully."

conda deactivate

# JOB-ID: 52666533
# bash script file name: /bash_scripts/DR_kraken2_pluspfdb-mcavraw.sh

#### Bracken

In [ ]:
salloc -c 4 -p cpu -t 3:00:00
## INSTALLATION
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/kraken2
conda install bracken

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=25G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 1:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-bracken-mcav-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/kraken2

SAMPLENAME="mcav"
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/ID_tables"
SAMPLELIST="DR_${SAMPLENAME}_sampleids.txt"
DBNAME="/datasets/bio/kraken2/PlusPF"
OUTDIR="/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/kraken2_pluspf/bracken"
mkdir -p $OUTDIR

THREADS=20
KMER_LEN=35 #(default)
READ_LEN=150

# Generate bracken database - just do once
bracken-build -d ${DBNAME} -t ${THREADS} -k ${KMER_LEN} -l ${READ_LEN} 

#${KRAKEN_DB}`  = location of the built Kraken 2 database
#${THREADS}`    = number of threads to use with Kraken and the Bracken scripts
#${KMER_LEN}`   = length of kmer used to build the Kraken database 
    #Kraken 1/KrakenUniq default kmer length = 31
    #Kraken 2 default kmer length = 35
    #Default set in the script is 35. 
#${READ_LEN}`   = the read length of your data e.g., if you are using 100 bp reads, set it to `100`. 
#need taxo.k2d, hash.k2d, and opts.k2d. in kraken2 database

# Abundance Estimation with Bracken
KRAKENFILES="/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/kraken2_pluspf/${SAMPLENAME}"
LEVEL1="S"
LEVEL2="P"

while IFS= read -r line; do
    # Execute bracken command for each file
    bracken -d "$DBNAME" -i $KRAKENFILES/"$line".kreport2 -o $OUTDIR/"${line%.kreport2}_$LEVEL1.bracken" -r "$READ_LEN" -l "$LEVEL1"
    bracken -d "$DBNAME" -i $KRAKENFILES/"$line".kreport2 -o $OUTDIR/"${line%.kreport2}_$LEVEL2.bracken" -r "$READ_LEN" -l "$LEVEL2"
if [ $? -eq 0 ]; then
        echo "bracken completed successfully for sample: $SAMPLEID"
    else
        echo "bracken encountered an error for sample: $SAMPLEID"
        exit 1
    fi
    done < "$LISTPATH/${SAMPLELIST}"
    
conda deactivate

#${SAMPLE}.kreport - the kraken report generated for a given dataset
#{BRACKEN_OUTPUT_FILE}.bracken - the desired name of the output file to be generated by the code
#The following optional parameters may be specified:
    #${LEVEL} - Default = 'S'. This specifies that abundance estimation will calculate estimated reads for each species. Other possible options are K (kingdom level), P (phylum), C (class), O (order), F (family), and G (genus).
    #${THRESHOLD} - Default = 10. For species classification, any species with <= 10 (or otherwise specified) reads will not receive any additional reads from higher taxonomy levels when distributing reads for abundance estimation. 
    #If another classification level is specified, thresholding will occur at that level.

# JOB-ID: 52247107
# bash script file name: /bash_scripts/bracken-mcav.sh 

job IDs for others \
dcyl: 52249829 \
mmea: 52249834 \
pstr: 52249836 \
dlab: 52249832 \
ssid: 52249833

*should have included family level in the initial script too, if I end up needing it, will re-run and add to the script above*

When looking at the Bracken outputs, a majority of these reads are human (tax ID 9606) even though I  ran reads through fastq-screen. I will filter out human assigned reads from the bracken files https://github.com/jenniferlu717/KrakenTools/blob/master/README.md using filter_bracken_out.py. This will also recalculate the sample fractions. \
*for phyla tax id*= 7711 \
*for species tax id*=9606

**make it easier on yourself and have the slurm output just have one level (just species or just phylum) so that normalizing is easier in the next step. Learned from trying to run all in same script and then not knowing enough reg ex to figure out how to parse the slurm the way I need**

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=10G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 1:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-bracken-filtering-species-%j.out  # %j = job ID  # %j = job ID

BRACKEN="/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/kraken2_pluspf/bracken"
OUTDIR="/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/kraken2_pluspf/bracken_filtered"
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/ID_tables"
SAMPLELIST="all_sampleids.txt"

#make sure filter_bracken.py script is in /scripts
while IFS= read -r SAMPLEID; do
python filter_bracken.out.py -i $BRACKEN/"${SAMPLEID}"_S.bracken -o $OUTDIR/"${SAMPLEID}"_S_filtered.bracken --exclude 9606
if [ $? -eq 0 ]; then
        echo "bracken filter successful for sample: $SAMPLEID"
    else
        echo "encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done < "$LISTPATH/${SAMPLELIST}"

echo "Bracken filter: All samples processed successfully."

#JOB-ID: 52470653
#bash script file name: bracken_filtering_species.sh

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=10G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 1:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-bracken-filtering-phylum-%j.out  # %j = job ID  # %j = job ID

BRACKEN="/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/kraken2_pluspf/bracken"
OUTDIR="/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/kraken2_pluspf/bracken_filtered"
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/ID_tables"
SAMPLELIST="all_sampleids.txt"

#make sure filter_bracken.py script is in /scripts
while IFS= read -r SAMPLEID; do
python filter_bracken.out.py -i $BRACKEN/"${SAMPLEID}"_P.bracken -o $OUTDIR/"${SAMPLEID}"_P_filtered.bracken --exclude 7711
if [ $? -eq 0 ]; then
        echo "bracken filter successful for sample: $SAMPLEID"
    else
        echo "encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done < "$LISTPATH/${SAMPLELIST}"

echo "Bracken filter: All samples processed successfully."

#JOB-ID: 52470654
#bash script file name: bracken_filtering_phylum.sh

In [ ]:
#To merge tables, I downloaded combine_bracken_outputs.py and put in the same directory as the bracken_filtered output
python combine_bracken_outputs.py --files *S_filtered.bracken --output DR_merged_all_bracken_species.txt
python combine_bracken_outputs.py --files *P_filtered.bracken --output DR_merged_all_bracken_phyla.txt

need to make a taxonomy table for phyloseq object

In [ ]:
#use make_ktaxonomy.py from KrakenTools
python make_ktaxonomy.py --nodes /datasets/bio/kraken2/PlusPF/nodes.dmp --names /datasets/bio/kraken2/PlusPF/names.dmp --seqid2taxid /datasets/bio/kraken2/PlusPF/seqid2taxid.map -o taxonomy.tsv